# 53 순차 검색의 답변 입력에 DB 사실 추가

사전 선택한12질문×두프로필×A/B×두회차=96조건. 52번 순차 검색·원질문·카드순위·U근거를 유지한다. A는 U근거만, B는 동일 카드의 DB일치사실을 답변 입력에만 추가한다. 원근거를 제거하지 않고64KB를지킨다. 미적용도 전체분모에남긴다. 동일회차에서 payload와근거소유권까지같은 A/B만 답변을공유하고, 회차간·입력이다른경우는공유하지않는다. HTML은직접검토후작성한다.

In [ ]:
from pathlib import Path
import importlib.util,sys,json,os,time,resource,copy
ROOT=Path.cwd()
if not (ROOT/'notebooks').exists():ROOT=ROOT.parent
sys.dont_write_bytecode=True
spec=importlib.util.spec_from_file_location('modes53',ROOT/'notebooks/scripts/52_retrieval_modes.py')
M=importlib.util.module_from_spec(spec);sys.modules[spec.name]=M;spec.loader.exec_module(M)
X=M.X;P=M.P;Q=M.Q
recovery=X.load('budget_recovery53','51_budget_recovery.py');recovery.install(X)
C=X.load('corrected_predicate53','52_query_predicate_correction.py');C.install(X)
PATH=P.F.STAGES[53];PATH.mkdir(exist_ok=True);P.F.initialize(53)
assert P.F.read_json(P.F.STAGES[52]/'completion_status.json')['stage53_allowed']
preselected=P.F.read_json(P.F.STAGES[50]/'53_preselected_questions.json')['query_ids']
queries=[q for q in P.F.read_json(P.F.STAGES[50]/'questions.json')['queries'] if q['query_id'] in preselected]
assert len(queries)==12
plans=P.F.read_json(P.F.STAGES[52]/'rerouted_repeat1_plans.json');by_query={r['parser']['query_id']:r for r in plans['records']}
prior=P.F.read_json(P.F.STAGES[52]/'execution_results.json')
bases={(r['query_id'],r['profile']):r for r in prior['retrievals'] if r['mode']=='SEQ' and r['query_id'] in preselected}
prior_manifest=P.F.read_json(P.F.STAGES[52]/'execution_manifest.json')
for q in queries:
 for pf in X.PROFILES:M.bind_retrieval(52,bases[q['query_id'],pf],question=q,profile=pf,mode='SEQ',parser_record=by_query[q['query_id']],manifest=prior_manifest)
import fitz
source_inputs={'data/raw/'+d['document_id']+'.pdf':d['source_pdf_sha256'] for d in Q.documents()}
source_page_counts={d['document_id']:len(fitz.open(ROOT/('data/raw/'+d['document_id']+'.pdf'))) for d in Q.documents()}
source_inputs.update({str((P.DATA/'30_structured_benefit_db'/name).relative_to(ROOT)):P.sha((P.DATA/'30_structured_benefit_db'/name).read_bytes()) for name in ['structured.sqlite','documents.jsonl','facts.jsonl']})
for name,wanted in source_inputs.items():assert P.sha((ROOT/name).read_bytes())==wanted
source_annotations=P.F.read_json(P.F.STAGES[52]/'frozen_annotations.json')
assert P.sha(source_annotations)==prior_manifest['annotations_sha256']
gold_by_query={r['query_id']:r['gold_cards'] for r in source_annotations['queries']}
identities=[f"{q['query_id']}:{pf}:{mode}:r{r}" for r in [1,2] for q in queries for pf in X.PROFILES for mode in ['A','B']]
manifest=X.make_manifest(53,ROOT/'notebooks/53_sequential_fact_supplement.ipynb',queries,dict(parser=[],answer=identities),maximum_answer=192,stage_cap='.8',extra=dict(source_sha256=prior_manifest['source_sha256'],split='development',logical_answer_conditions=96,
 source_input_sha256=source_inputs,source_page_counts=source_page_counts,upstream_annotations_sha256=P.sha(source_annotations),
 preselection_sha256=P.sha(P.F.read_json(P.F.STAGES[50]/'53_preselected_questions.json')),upstream_result_sha256=P.sha(prior),upstream_parser_sha256=P.sha(plans),
 decision='B only >=1 question PASS-frequency win; 0 losses/new important errors/condition loss/scope errors; otherwise A',unchanged_U_evidence=True,append_DB_facts_only_to_answer=True,keep_inapplicable_cases_in_denominator=True))
X.frozen(PATH/'questions.json',dict(queries=queries,split='development'))
ALLOW_NETWORK=os.getenv('PAID_RETEST_ALLOW_NETWORK')=='1'
print('96답변조건. 신규 검색·임베딩·재정렬 없음.')

In [ ]:
from pickcardu_rag.answering import serialize_answer_payload
started=time.perf_counter();cpu_started=time.process_time();retrievals=[];answers=[]
existing_answer_ids={p.stem for p in (PATH/"answers").glob("*.json")}
initial_api_keys={p.stem for p in (PATH/"api_cache").glob("*.json")}
for repeat in [1,2]:
 for qi,q in enumerate(queries):
  record=by_query[q['query_id']]
  for pf in X.PROFILES:
   base=bases[q['query_id'],pf];sides={}
   for mode in ['A','B']:
    identity=f"{q['query_id']}:{pf}:{mode}:r{repeat}";target=PATH/'retrieval'/(identity+'.json');new=not target.exists()
    extra=dict(upstream_SEQ_sha256=P.sha(base),intervention='unchanged U' if mode=='A' else 'append whole matching DB facts to answer only')
    if not new:row=P.F.read_json(target)
    else:
     result=copy.deepcopy(base['result']) if mode=='A' else M.supplement(q['question'],base['result'],record['routing'])
     row=dict(identity=identity,query_id=q['query_id'],profile=pf,mode=mode,repeat=repeat,result=result,route=record['routing'],original_scope=record['original_scope'],manifest_sha256=P.sha(manifest))
    M.bind_retrieval(53,row,question=q,profile=pf,mode=mode,parser_record=record,manifest=manifest,repeat=repeat,extra_inputs=extra,new=new)
    if new:P.F.write_json(target,row)
    retrievals.append(row);sides[mode]=row
   equal=serialize_answer_payload(q['question'],sides['A']['result']['evidence'])==serialize_answer_payload(q['question'],sides['B']['result']['evidence'])
   equal=equal and P.sha(sides['A']['result']['evidence'])==P.sha(sides['B']['result']['evidence'])
   order=['A','B'] if equal or (repeat+qi)%2 else ['B','A'];paired={}
   for mode in order:
    row=sides[mode];extra=dict(upstream_SEQ_sha256=P.sha(base),intervention='unchanged U' if mode=='A' else 'append whole matching DB facts to answer only')
    M.bind_retrieval(53,row,question=q,profile=pf,mode=mode,parser_record=record,manifest=manifest,repeat=repeat,extra_inputs=extra)
    reused=paired.get('A') if mode=='B' and equal else None
    a=M.answer(53,row['identity'],q['question'],row['result'],allow_network=ALLOW_NETWORK,reused=reused);paired[mode]=a
    answers.append({**a,'query_id':q['query_id'],'profile':pf,'mode':mode,'repeat':repeat,'identical_AB_input':equal})
    P.F.write_json(PATH/'progress.json',dict(completed=len(answers),expected=96,last_identity=row['identity'],ledger=X.ledger()))
    print('답변',row['identity'],a['status'],'동일입력공유' if reused else '독립조건',round(a['answer_ready_wall_seconds'],3),'초',flush=True)
segment=dict(wall_seconds=time.perf_counter()-started,cpu_seconds=time.process_time()-cpu_started,
 peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,existing_answer_count_at_start=len(existing_answer_ids),
 newly_created_answer_count=len({a['identity'] for a in answers}-existing_answer_ids),
 new_physical_requests=len({p.stem for p in (PATH/'api_cache').glob('*.json')}-initial_api_keys),
 time_scope='this execution segment only; saved logical API timings remain first-write values',
 new_embeddings=0,new_searches=0,new_reranker_pairs=0,gpu='미사용')
P.F.write_json(PATH/'latest_execution_segment.json',segment)
resource_journal=PATH/'execution_segments'
resource_journal.mkdir(exist_ok=True)
P.F.write_json(resource_journal/(str(time.time_ns())+'.json'),segment)
completed=dict(answers=answers,retrievals=retrievals,manifest_sha256=P.sha(manifest),
 first_completed_execution_segment=segment,api_ledger=X.ledger())
result_path=PATH/'execution_results.json'
if result_path.exists():
 saved=P.F.read_json(result_path)
 assert saved['answers']==answers and saved['retrievals']==retrievals and saved['manifest_sha256']==P.sha(manifest)
 print('최초 완료 결과·자원은 보존하고 이번 재검사 시간만 별도 저장한다.')
else:P.F.write_json(result_path,completed)
P.F.write_json(PATH/'completion_status.json',dict(status='awaiting_source_evaluation',logical_conditions=len(answers),expected=96,stage54_allowed=False))
print('실행 완료. 전체12문항을 분모로 원문 평가 후 A/B를 선정한다.')

## 원문 평가와 선정

전체12문항을 유지하고, 같은 질문의2회 PASS 횟수를 비교한다. 각 프로필에서 개선이1문항 이상이며 악화·새 중요오류·조건손실·범위위반이 모두0일 때만B를 선정한다. 실제 판정은 결과를 원문과 읽어 별도JSON에 기록하며 아래 셀이 그 판정의 분모와 선택 규칙을 검사한다.

In [ ]:
# 원문 판정은 API에 보내지 않는다. 전체 논리 분모를 유지해 프로필별 A/B를 선정한다.
judgment_path=PATH/'source_judgments.json'
if not judgment_path.exists():
 print('96조건 원문 판정 대기. 현재 선정 없음.')
else:
 completed=P.F.read_json(PATH/'execution_results.json')
 assert completed['manifest_sha256']==P.sha(manifest)
 for name,wanted in manifest['source_input_sha256'].items():assert P.sha((ROOT/name).read_bytes())==wanted
 retrieval_by_id={r['identity']:r for r in completed['retrievals']}
 rows=completed['answers'];annotations=P.F.read_json(judgment_path)
 judged={r['identity']:r for r in annotations['records']}
 assert len(rows)==len(judged)==96 and set(judged)=={r['identity'] for r in rows}
 for row in rows:
  j=judged[row['identity']]
  assert j['answer_sha256']==P.sha(row) and j['reason'] and j['source_support']
  assert j['verdict'] in ['PASS','PARTIAL','FAIL','UNCERTAIN']
  assert all(type(j[k]) is bool for k in ['important_error','condition_loss_vs_A','scope_error'])
  assert j['question_sha256']==P.sha(next(q['question'] for q in queries if q['query_id']==row['query_id']))
  retrieved={r['card_key'] for r in retrieval_by_id[row['identity']]['result']['cards']+retrieval_by_id[row['identity']]['result']['evidence']}
  for support in j['source_support']:
   key=support['card_key'];name='data/raw/'+key+'.pdf'
   assert support['path']==name and support['sha256']==manifest['source_input_sha256'][name]
   assert type(support['page']) is int and 1<=support['page']<=manifest['source_page_counts'][key]
   assert isinstance(support['observation'],str) and support['observation'].strip()
   assert support['relation'] in ['retrieved_card','missing_expected_card']
   if support['relation']=='retrieved_card':assert key in retrieved
   else:assert j['verdict']!='PASS' and key in gold_by_query[row['query_id']]
  if row['status']!='ok':assert j['verdict']!='PASS'
 decisions=[]
 for pf in X.PROFILES:
  pairs=[];new_errors=[];losses=[];scope_errors=[]
  for q in queries:
   counts={m:0 for m in ['A','B']}
   for repeat in [1,2]:
    pair={m:judged[f"{q['query_id']}:{pf}:{m}:r{repeat}"] for m in ['A','B']}
    answer_pair={m:next(r for r in rows if r['identity']==pair[m]['identity']) for m in ['A','B']}
    if answer_pair['B']['identical_AB_input']:
     assert answer_pair['B']['reuse_source_sha256']==P.sha(P.F.read_json(PATH/'answers'/(pair['A']['identity']+'.json')))
     assert all(pair['A'][k]==pair['B'][k] for k in ['verdict','important_error','scope_error'])
     assert not pair['B']['condition_loss_vs_A']
    for m in ['A','B']:counts[m]+=pair[m]['verdict']=='PASS'
    if pair['B']['important_error'] and not pair['A']['important_error']:new_errors.append(pair['B']['identity'])
    if pair['B']['condition_loss_vs_A']:losses.append(pair['B']['identity'])
    if pair['B']['scope_error']:scope_errors.append(pair['B']['identity'])
   pairs.append(dict(query_id=q['query_id'],A_passes=counts['A'],B_passes=counts['B'],
    outcome='win' if counts['B']>counts['A'] else 'loss' if counts['B']<counts['A'] else 'tie'))
  wlt={k:sum(p['outcome']==k for p in pairs) for k in ['win','loss','tie']}
  selected='B' if wlt['win']>=1 and not (wlt['loss'] or new_errors or losses or scope_errors) else 'A'
  decisions.append(dict(profile=pf,selected=selected,question_denominator=12,repeats=2,
   logical_denominator_per_arm=24,A_passes=sum(p['A_passes'] for p in pairs),B_passes=sum(p['B_passes'] for p in pairs),
   question_wins_losses_ties=wlt,new_important_error_ids=new_errors,condition_loss_ids=losses,scope_error_ids=scope_errors,question_pairs=pairs))
 summary=dict(split='development',judgment_sha256=P.sha(annotations),execution_results_sha256=P.sha(completed),
  decisions=decisions,selection_rule=manifest['decision'],independent_human_review=False)
 X.frozen(PATH/'evaluation_summary.json',summary)
 print(json.dumps(decisions,ensure_ascii=False,indent=2))
